# 02 · DDAS toy sampler

cluster 희소성과 난도를 함께 가중해 sampling 분포를 바꾼다. 실제 65.5M dataset이나 저자 sampling policy를 복제하지 않는다.

**학습 목표**: cluster 빈도와 난도 weight를 결합한 DDAS식 sampling이 선택 빈도를 어떻게 바꾸는지 관찰한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `random`, `collections`, `math`만 사용하며 외부 패키지는 없다.

In [ ]:
# sqrt 역가중은 희귀 cluster를 강조하면서 단순 역빈도보다 과도한 증폭을 줄인다.
import random
from collections import Counter
from math import sqrt

items = [
    {'id': 'a1', 'cluster': 'text', 'difficulty': 'Easy'},
    {'id': 'a2', 'cluster': 'text', 'difficulty': 'Easy'},
    {'id': 'a3', 'cluster': 'text', 'difficulty': 'Medium'},
    {'id': 'b1', 'cluster': 'table', 'difficulty': 'Hard'},
    {'id': 'c1', 'cluster': 'formula', 'difficulty': 'Medium'},
]
difficulty_weight = {'Easy': 1.0, 'Medium': 1.8, 'Hard': 2.5}
cluster_size = Counter(item['cluster'] for item in items)

def ddas_weight(item):
    diversity = 1 / sqrt(cluster_size[item['cluster']])
    return min(3.0, diversity * difficulty_weight[item['difficulty']])

def sample_ids(n, seed=17):
    rng = random.Random(seed)
    return rng.choices([x['id'] for x in items], weights=[ddas_weight(x) for x in items], k=n)


In [ ]:
print('weights:')
for item in items:
    print(item['id'], item['cluster'], item['difficulty'], round(ddas_weight(item), 3))
counts = Counter(sample_ids(5000))
print('sample frequencies:', {key: round(value / 5000, 3) for key, value in sorted(counts.items())})
assert counts['b1'] > counts['a1']


희귀·Hard 예제를 강조하되 noise도 함께 과표집될 수 있다. 실제 data engine에는 품질 gate, weight cap, source별 leakage 감사와 epoch별 분포 기록이 필요하다.